# 🎧 有聲書批次轉 Markdown (Google Colab 免費 GPU 極速版 - 低記憶體保護)

本筆記本已加入 **低記憶體串流分塊 (15 分鐘 Chunking)**，徹底解決 5~20 小時超長篇有聲書導致 Colab 記憶體爆滿 (RAM Crashed) 的問題！

### ⚠️ 使用前確認
1. **開啟 GPU**：看頂部按鈕點選 `▷ 全部執行 ▾` 旁的箭頭 ➔ **「變更執行階段類型」➔ 硬體加速器選擇「T4 GPU」**。
2. **直接點「全部執行」**即可！

### 步驟 1：安裝極速轉錄套件

In [ ]:
!pip install -q faster-whisper tqdm av

### 步驟 2：掛載 Google 雲端硬碟 (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 步驟 3：鎖定「2022-0708/audiobook」並以低記憶體串流模式高速轉錄

In [ ]:
#@title ⚙️ 轉錄參數設定 { run: "auto" }
TARGET_FOLDER = "2022-0708/audiobook" #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/audiobook_md" #@param {type:"string"}
MODEL_NAME = "small.en" #@param ["base.en", "small.en", "medium.en", "large-v3"]
INTERVAL_MINS = 5 #@param {type:"integer"}

import os
import av
import numpy as np
from pathlib import Path
from datetime import timedelta, datetime
from faster_whisper import WhisperModel
from tqdm.auto import tqdm

audio_exts = (".mp3", ".m4a", ".m4b", ".wav", ".aac", ".flac")
audio_files = []

def get_duration(file_path):
    try:
        with av.open(str(file_path)) as c:
            if c.duration:
                return float(c.duration) / av.time_base
            for s in c.streams.audio:
                if s.duration and s.time_base:
                    return float(s.duration * s.time_base)
    except Exception:
        pass
    return 0.0

def stream_chunks(file_path, chunk_sec=900.0, sample_rate=16000):
    """15 分鐘串流載入，嚴格鎖定 RAM 在 150MB 以下，絕不爆 Colab 記憶體"""
    container = av.open(str(file_path))
    resampler = av.audio.resampler.AudioResampler(format="s16", layout="mono", rate=sample_rate)
    chunk_samples = int(chunk_sec * sample_rate)
    buf = []
    cnt = 0
    t = 0.0

    for frame in container.decode(audio=0):
        for rframe in resampler.resample(frame):
            arr = rframe.to_ndarray().flatten()
            buf.append(arr)
            cnt += len(arr)
            if cnt >= chunk_samples:
                full = np.concatenate(buf)
                chunk = full[:chunk_samples].astype(np.float32) / 32768.0
                clen = len(chunk) / sample_rate
                yield t, chunk, clen
                t += clen
                buf = [full[chunk_samples:]]
                cnt = len(buf[0])

    if buf and cnt > 0:
        full = np.concatenate(buf).astype(np.float32) / 32768.0
        clen = len(full) / sample_rate
        yield t, full, clen

# 搜尋目錄
target_pattern = TARGET_FOLDER.replace("\\", "/").strip("/").lower()
print(f"🔍 正在為您精準鎖定包含「{TARGET_FOLDER}」的資料夾...")
search_roots = [Path("/content/drive/Othercomputers"), Path("/content/drive/MyDrive"), Path("/content/drive")]

for s_root in search_roots:
    if not s_root.exists():
        continue
    for root, dirs, files in os.walk(s_root):
        nroot = root.replace("\\", "/").lower()
        if nroot.endswith(target_pattern) or target_pattern in nroot:
            found = [f for f in files if Path(f).suffix.lower() in audio_exts]
            if found:
                audio_files = [Path(root) / f for f in found]
                print(f"🎯 成功鎖定目標資料夾: {root} (共 {len(found)} 本有聲書)")
                break
    if audio_files:
        break

if not audio_files:
    print(f"\n❌ 未找到路徑符合「{TARGET_FOLDER}」的資料夾，請檢查 Colab 左側 drive/Othercomputers 是否已連線。")
else:
    audio_files = sorted(audio_files)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"\n正在載入 GPU 語音模型 ({MODEL_NAME})...")
    model = WhisperModel(MODEL_NAME, device="cuda", compute_type="float16")

    def fmt_time(seconds):
        td = timedelta(seconds=int(seconds))
        s = int(td.total_seconds())
        return f"{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}"

    for idx, audio_path in enumerate(audio_files, 1):
        title = audio_path.stem
        for noise in ["[ ezmp3.cc ]", "[ ezmp3.co ]", "_ ezmp3.cc", "(ezmp3.cc)"]:
            title = title.replace(noise, "").strip()

        out_md = Path(OUTPUT_DIR) / f"{title}.md"
        if out_md.exists():
            print(f"[{idx:02d}/{len(audio_files):02d}] ⏩ 跳過已完成: {title}")
            continue

        total_dur = get_duration(audio_path)
        print(f"\n[{idx:02d}/{len(audio_files):02d}] 🎧 正在轉錄: {title} (總長: {fmt_time(total_dur)})")
        sections = []
        curr_sec_start = 0.0
        curr_texts = []
        sec_interval_s = INTERVAL_MINS * 60.0
        last_pbar_time = 0.0

        with tqdm(total=int(total_dur) if total_dur > 0 else 100, unit="s", desc=title[:20]) as pbar:
            for chunk_offset, chunk_audio, chunk_len in stream_chunks(audio_path, chunk_sec=900.0):
                segs, _ = model.transcribe(chunk_audio, beam_size=5, vad_filter=True)
                for s in segs:
                    st = s.text.strip()
                    if not st:
                        continue
                    abs_s = chunk_offset + s.start
                    abs_e = chunk_offset + s.end
                    
                    if total_dur > 0:
                        adv = max(0, int(abs_e - last_pbar_time))
                        if adv > 0:
                            pbar.update(adv)
                            last_pbar_time = abs_e
                    
                    if abs_s >= curr_sec_start + sec_interval_s and curr_texts:
                        sections.append({"start": curr_sec_start, "end": abs_s, "text": " ".join(curr_texts)})
                        curr_sec_start = abs_s
                        curr_texts = [st]
                    else:
                        curr_texts.append(st)

            if curr_texts:
                sections.append({"start": curr_sec_start, "end": total_dur if total_dur > 0 else (chunk_offset + chunk_len), "text": " ".join(curr_texts)})
            if total_dur > last_pbar_time:
                pbar.update(int(total_dur - last_pbar_time))

        with open(out_md, "w", encoding="utf-8") as f:
            f.write(f"# {title}\n\n")
            f.write(f"> **時長**：`{fmt_time(total_dur)}` | **模型**：`{MODEL_NAME}` | **日期**：`{datetime.now().strftime('%Y-%m-%d')}`\n\n")
            f.write("## 目錄 (Table of Contents)\n\n")
            for s in sections:
                st_str = fmt_time(s['start'])
                en_str = fmt_time(s['end'])
                f.write(f"- [{st_str} - {en_str}](#sec-{st_str.replace(':', '')})\n")
            f.write("\n---\n\n## 逐字稿與筆記\n\n")
            for s in sections:
                st_str = fmt_time(s['start'])
                en_str = fmt_time(s['end'])
                f.write(f'<a id="sec-{st_str.replace(":", "")}"></a>\n')
                f.write(f"### ⏱️ [{st_str} - {en_str}]\n\n{s['text']}\n\n")

        print(f"✅ 完成並存檔: {out_md}")

    print("\n🎉 全部目標有聲書已全數轉錄完成！")